# Colab: real-data PINN, batch 3 (zoned designs + canal seepage)

**What batch 2 showed.**
- The best regional setting (chosen by the rule fixed in advance) is `tb_off1`: terrain + S in the report range + weaker well corrections.
- One network *per zone* goes badly wrong at places with no well (19 m error vs 9.5 m regional), because each zone network sees only its own 73–269 wells.

**What this batch does.**
1. Re-runs the 4 zone networks with the chosen setting (fair comparison).
2. Runs the planned fix (PLAN.md E20, "shared"): **one** network for the water level using all wells, with storage, transmissivity and pumping set **per zone**.
3. Adds **canal seepage** (water leaking from canals into the ground). Near canals the model guessed the water level about 6 m too low, because recharge was rain only. Main strength = the GEC-2015 norm; half and double are sensitivity checks.

The final runs (5 seeds × 3 layouts) come in batch 4, once these settings are decided.

Everything is scored on **validation** wells only. The test wells stay sealed.

**Before you start:**
1. Upload the new `pinn_code.zip` (from the laptop's `pinn` folder) to `MyDrive/pinn/`, replacing the old one.
2. Choose **Runtime → Change runtime type → T4 GPU**, then **Runtime → Run all**.

**Timing.** 9 runs of about 20 minutes (about 3 hours). **The first 3 runs (about 1 hour) answer the main questions.** If you are short on time, run the last cell whenever you stop and send me the zip; you can continue the rest later. Runs that already finished (also from an earlier start of this batch) are skipped.

**Keep the laptop awake** (it must not sleep; the screen may turn off). If Colab disconnects anyway, just do **Run all** again. Finished runs are skipped.

In [ ]:
DRIVE_ROOT = "/content/drive/MyDrive/pinn"

In [ ]:
import glob
import os
import shutil
import zipfile
from google.colab import drive

drive.mount("/content/drive")
os.makedirs(f"{DRIVE_ROOT}/runs/logs", exist_ok=True)
CODE = "/content/pinn"
shutil.rmtree(CODE, ignore_errors=True)
with zipfile.ZipFile(f"{DRIVE_ROOT}/pinn_code.zip") as z:
    z.extractall(CODE)
%cd {CODE}

In [ ]:
# PyTorch is already installed on Colab; this adds everything else (about 1 minute).
!pip install -q -r requirements.txt
!pip install -q -e .
os.environ["PINN_RUNS_DIR"] = f"{DRIVE_ROOT}/runs"
!python scripts/check_env.py
!python -m pytest -q -m "not data"

## Runs (most important first)

| Runs | Question it answers |
|---|---|
| shared `tb_off1`, layout 0 | one network, physics per zone: does sharing the water-level surface fix the unseen places? |
| regional and shared `tbc` (canal seepage, GEC norm) | does adding canal water help, and does zoning gain from it? |
| zones 1–4 `tb_off1`, layout 0 | the per-zone method with the chosen setting (fair comparison) |
| regional `tbc05`, `tbc2` (half / double canal seepage) | how sensitive the result is to the canal assumption |

**Rules (fixed before these runs):** score = validation error on task A / best baseline + task B / best baseline (as before).
- Canal seepage is kept if regional `tbc` scores lower than regional `tb_off1`. Half and double are reported, not chosen from.
- Zoned design: per-zone vs shared, the lower score (with canals if kept). The winner gets the final runs in batch 4.

In [ ]:
def done(pattern):
    return bool(glob.glob(f"{DRIVE_ROOT}/runs/{pattern}/metrics.json"))

LOG = f"{DRIVE_ROOT}/runs/logs/real3.log"
RUNS = [
    ("*_real_L0_kmeansk5_shared_s0_tb_off1", "--shared --tag tb_off1"),
    ("*_real_L0_regional_s0_tbc", "--regional --tag tbc --set canal_seepage.strength=1"),
    ("*_real_L0_kmeansk5_shared_s0_tbc", "--shared --tag tbc --set canal_seepage.strength=1"),
    *[(f"*_real_L0_kmeansk5_z{z}_s0_tb_off1", f"--zone {z} --tag tb_off1") for z in (1, 2, 3, 4)],
    ("*_real_L0_regional_s0_tbc05", "--regional --tag tbc05 --set canal_seepage.strength=0.5"),
    ("*_real_L0_regional_s0_tbc2", "--regional --tag tbc2 --set canal_seepage.strength=2"),
]
for pattern, args in RUNS:
    if done(pattern):
        print("already done:", pattern)
        continue
    !python -u scripts/run_real.py {args} --stage-only val 2>&1 | tee -a {LOG}

## Bundle the results for Claude
This makes `MyDrive/pinn/real3_results.zip` with every **finished** run so far (you can run this cell at any time). Download it, put it in the `pinn` folder on the laptop, then tell Claude.

In [ ]:
out = f"{DRIVE_ROOT}/real3_results.zip"
n = 0
with zipfile.ZipFile(out, "w", zipfile.ZIP_DEFLATED) as z:
    for run in glob.glob(f"{DRIVE_ROOT}/runs/*_real_*"):
        if not run.endswith(("_tb_off1", "_tbc", "_tbc05", "_tbc2")) or not os.path.exists(f"{run}/metrics.json"):
            continue
        n += 1
        for path in glob.glob(f"{run}/*"):
            z.write(path, os.path.relpath(path, f"{DRIVE_ROOT}/runs"))
    if os.path.exists(LOG):
        z.write(LOG, "logs/real3.log")
print("wrote", out, f"({n} finished runs, {os.path.getsize(out) / 1e6:.2f} MB)")